## Etapa 1: Aquisição (RGB)

In [4]:
import os
import cv2
import numpy as np

def carregar_imagem(caminho):
    with open(caminho, 'rb') as f:
        dados = np.frombuffer(f.read(), np.uint8)
    img_bgr = cv2.imdecode(dados, cv2.IMREAD_COLOR)
    if img_bgr is None:
        return None
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

base_path = r"E:\VISÃO COMPUTACIONAL\Projeto Final\DATASET\normal"
classes = ["fresh", "rotten"]

imagens_por_classe = {}
for classe in classes:
    pasta_classe = os.path.join(base_path, classe)
    imagens = []
    for raiz, subpastas, arquivos in os.walk(pasta_classe):
        for arquivo in arquivos:
            if arquivo.lower().endswith(('.png', '.jpg', '.jpeg')):
                imagens.append(os.path.join(raiz, arquivo))
    imagens.sort()
    imagens_por_classe[classe] = imagens
    print(f"Classe '{classe}': {len(imagens)} imagens encontradas.")

Classe 'fresh': 1143 imagens encontradas.
Classe 'rotten': 1143 imagens encontradas.


## Etapa 2: Pré-processamento (Filtro e Realce)

In [ ]:
def carregar_imagem_bgr(caminho):
    with open(caminho, 'rb') as f:
        dados = np.frombuffer(f.read(), np.uint8)
    img = cv2.imdecode(dados, cv2.IMREAD_COLOR)
    return img

def preprocessar(img_bgr):
    img_suave = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    img_hsv = cv2.cvtColor(img_suave, cv2.COLOR_BGR2HSV)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    h, s, v = cv2.split(img_hsv)
    v_eq = clahe.apply(v)
    img_hsv_eq = cv2.merge([h, s, v_eq])
    return cv2.cvtColor(img_hsv_eq, cv2.COLOR_HSV2BGR)

def salvar_imagem(caminho_destino, img_bgr):
    ext = os.path.splitext(caminho_destino)[1]
    ret, buffer = cv2.imencode(ext, img_bgr)
    if not ret:
        return False
    with open(caminho_destino, 'wb') as f:
        f.write(buffer.tobytes())
    return True

base_path = r"E:\VISÃO COMPUTACIONAL\Projeto Final\DATASET"
output_base = r"E:\VISÃO COMPUTACIONAL\Projeto Final\DATASET\preprocessed"

total_ok = 0
total_falhas = 0

for classe in classes:
    print(f"\nProcessando classe: {classe}")
    for i, caminho_original in enumerate(imagens_por_classe[classe], start=1):
        rel_path = os.path.relpath(caminho_original, base_path)
        caminho_destino = os.path.join(output_base, rel_path)
        
        os.makedirs(os.path.dirname(caminho_destino), exist_ok=True)
        
        img = carregar_imagem_bgr(caminho_original)
        if img is None:
            print(f"  [{i}/{len(imagens_por_classe[classe])}] ERRO ao carregar: {caminho_original}")
            total_falhas += 1
            continue
        
        img_proc = preprocessar(img)
        
        sucesso = salvar_imagem(caminho_destino, img_proc)
        if not sucesso:
            print(f"  [{i}/{len(imagens_por_classe[classe])}] ERRO ao salvar: {caminho_destino}")
            total_falhas += 1
        else:
            total_ok += 1
        
        if i % 100 == 0:
            print(f"  ... {i} imagens processadas em '{classe}'")

print(f"\nResumo final:")
print(f"  Imagens processadas com sucesso: {total_ok}")
print(f"  Falhas: {total_falhas}")
print(f"  Destino: {output_base}")


Processando classe: fresh
  ... 100 imagens processadas em 'fresh'
  ... 200 imagens processadas em 'fresh'
  ... 300 imagens processadas em 'fresh'
  ... 400 imagens processadas em 'fresh'
  ... 500 imagens processadas em 'fresh'
  ... 600 imagens processadas em 'fresh'
  ... 700 imagens processadas em 'fresh'
  ... 800 imagens processadas em 'fresh'
  ... 900 imagens processadas em 'fresh'
  ... 1000 imagens processadas em 'fresh'
  ... 1100 imagens processadas em 'fresh'

Processando classe: rotten
  ... 100 imagens processadas em 'rotten'
  ... 200 imagens processadas em 'rotten'
  ... 300 imagens processadas em 'rotten'
  ... 400 imagens processadas em 'rotten'
  ... 500 imagens processadas em 'rotten'
  ... 600 imagens processadas em 'rotten'
  ... 700 imagens processadas em 'rotten'
  ... 800 imagens processadas em 'rotten'
  ... 900 imagens processadas em 'rotten'
  ... 1000 imagens processadas em 'rotten'
  ... 1100 imagens processadas em 'rotten'

Resumo final:
  Imagens pro